# VigieMer — 02 · Qui est invisible ? Croisement radar × AIS

Le notebook 01 a trouvé les navires sur l'image radar. Mais un navire vu par le radar n'est pas forcément suspect : la plupart émettent leur position AIS normalement.

**Ici, on compare chaque détection aux positions AIS publiées par Global Fishing Watch au même moment.**
- Détection + AIS à proximité → navire **identifié** (nom, pavillon, type).
- Détection sans AIS → **navire « dark »**, candidat à une alerte.

**Avant de lancer :**
1. *Add Input* (panneau de droite) → *Your Work* → *Notebooks* → ajoute la dernière version du **notebook 01**.
2. *Add-ons → Secrets* : crée un secret nommé `GFW_TOKEN` avec ta clé Global Fishing Watch, et coche-le pour ce notebook.
3. *Internet : On*. Pas besoin de GPU ici.

## 1. Réglages

In [ ]:
# Distance maximale (km) entre un écho radar et une position AIS pour dire « c'est le même navire ».
# GFW donne une position par heure dans une case de 0,01° (≈ 1,1 km) : avec l'imprécision de la case,
# le balancement d'un navire au mouillage et l'erreur radar, 3 km reste prudent. Le diagnostic 5 bis
# montre qu'il n'y a pas de décalage systématique entre radar et AIS.
MATCH_KM = 3.0

# Fenêtre de temps autour du passage du satellite (heures) pour accepter une position AIS
TIME_WINDOW_H = 1

# En dessous de cette longueur, beaucoup de bateaux (pirogues, petits chalutiers) n'ont
# AUCUNE obligation d'AIS : on ne les classe pas en alerte prioritaire.
MIN_ALERT_LENGTH_M = 25

# Confiance minimale du détecteur pour garder une détection
MIN_SCORE = 0.9

# Objets fixes : même endroit (à 200 m près) sur au moins 3 passages différents
FIXED_M = 200
FIXED_MIN_PASSES = 3

OUT = "/kaggle/working/vigiemer_ais"

## 2. Charger les détections du notebook 01

In [ ]:
import os, glob
import pandas as pd

# On cherche tous les fichiers detections.csv dans les entrées du notebook
paths = glob.glob("/kaggle/input/**/detections.csv", recursive=True)

if not paths:
    # Diagnostic : on affiche ce que Kaggle a réellement monté en entrée
    print("Contenu de /kaggle/input :")
    found_any = False
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count(os.sep) - 2
        if depth > 4:
            continue
        print("  " * depth + os.path.basename(root) + "/")
        for f in files[:8]:
            print("  " * (depth + 1) + f)
            found_any = True
    if not found_any:
        print("  (vide) -> aucune entrée n'est attachée à ce notebook.")
    raise SystemExit(
        "Aucun detections.csv trouvé.\n"
        "1) Dans le notebook 01 : Save Version -> 'Save & Run All (Commit)'. Attends la fin.\n"
        "2) Ici : Add Input -> Your Work -> Notebooks -> notebook 01 -> Add.\n"
        "3) Relance ce notebook.")
dets = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)

# Dossier racine des sorties du notebook 01 (parent des dossiers de scène)
INPUT_ROOT = os.path.dirname(os.path.dirname(paths[0]))

# Fiches des scènes (heure, emprise au sol) écrites par le notebook 01
import json
scene_meta = {}
for p in glob.glob(os.path.join(INPUT_ROOT, "*", "scene_meta.json")):
    with open(p) as f:
        m = json.load(f)
    scene_meta[m["id"]] = m

dets["acquired"] = pd.to_datetime(dets["acquired"], utc=True)   # heure du passage satellite
dets = dets[(~dets["on_land"].astype(bool)) & (dets["score"] >= MIN_SCORE)].copy()
dets = dets.reset_index(drop=True)
os.makedirs(OUT, exist_ok=True)

print(f"{len(paths)} scènes, {len(dets)} détections en mer")
dets.groupby("scene_id").agg(n=("score", "size"), heure=("acquired", "first"))

## 3. Connexion à Global Fishing Watch

In [ ]:
import requests
from kaggle_secrets import UserSecretsClient

# La clé est lue depuis les secrets Kaggle : elle n'apparaît jamais dans le notebook
GFW_TOKEN = UserSecretsClient().get_secret("GFW_TOKEN")
HEADERS = {"Authorization": f"Bearer {GFW_TOKEN}"}
REPORT_URL = "https://gateway.api.globalfishingwatch.org/v3/4wings/report"
GFW_TOKEN = GFW_TOKEN.strip()                     # enlève espaces / retours à la ligne collés par erreur
# Une clé Global Fishing Watch est un long jeton (JWT) qui commence par "eyJ".
# Une clé qui commence par "KGAT_" est une clé KAGGLE : ce n'est pas la bonne.
if GFW_TOKEN.startswith("KGAT_"):
    raise SystemExit("Le secret GFW_TOKEN contient une clé Kaggle, pas une clé Global Fishing Watch. "
                     "Crée la clé sur https://globalfishingwatch.org/our-apis/tokens")
if not GFW_TOKEN.startswith("eyJ"):
    print("Attention : cette clé ne ressemble pas à une clé GFW (elle devrait commencer par 'eyJ').")
print(f"Clé chargée : {GFW_TOKEN[:4]}… ({len(GFW_TOKEN)} caractères)")

## 4. Télécharger l'AIS au moment de chaque image

On demande à GFW, pour la zone de chaque image, la présence AIS **heure par heure**, **navire par navire**, sur une grille de 0,01°.

In [ ]:
import io, json, zipfile, time

def flatten_entries(payload):
    """La réponse JSON range les lignes par jeu de données : on les met à plat."""
    rows = []
    for entry in payload.get("entries", []):
        if isinstance(entry, dict):
            for v in entry.values():           # clé = nom du dataset, valeur = liste
                if isinstance(v, list):
                    rows.extend(v)
        elif isinstance(entry, list):
            rows.extend(entry)
    return rows

def read_report(resp):
    """GFW renvoie soit du JSON, soit un zip contenant le JSON : on gère les deux."""
    if resp.content[:2] == b"PK":              # signature d'un fichier zip
        z = zipfile.ZipFile(io.BytesIO(resp.content))
        name = [n for n in z.namelist() if n.endswith(".json")][0]
        return json.loads(z.read(name))
    return resp.json()

def ais_presence(bbox, day):
    """Présence AIS horaire par navire dans un rectangle, pour un jour donné."""
    lon0, lat0, lon1, lat1 = bbox
    polygon = {"type": "Polygon",
               "coordinates": [[[lon0, lat0], [lon1, lat0], [lon1, lat1], [lon0, lat1], [lon0, lat0]]]}
    feature_collection = {"type": "FeatureCollection",
                          "features": [{"type": "Feature", "properties": {}, "geometry": polygon}]}
    params = {
        "datasets[0]": "public-global-presence:latest",   # présence AIS de tous les navires
        "date-range": f"{day:%Y-%m-%d},{day + pd.Timedelta(days=1):%Y-%m-%d}",
        "temporal-resolution": "HOURLY",
        "spatial-resolution": "HIGH",                     # cases de 0,01°
        "group-by": "VESSEL_ID",                          # une ligne par navire
        "format": "JSON",
    }
    # La documentation montre deux formes possibles pour la zone : on essaie les deux
    for body in ({"geojson": feature_collection}, {"geojson": polygon}):
        for attempt in range(4):
            r = requests.post(REPORT_URL, params=params, json=body, headers=HEADERS, timeout=180)
            if r.status_code == 429:                       # un seul rapport à la fois par clé
                time.sleep(20); continue
            break
        if r.ok:
            return pd.DataFrame(flatten_entries(read_report(r)))
        print(f"  essai refusé ({r.status_code}) : {r.text[:200]}")
    r.raise_for_status()

ais_frames = []
for sid, g in dets.groupby("scene_id"):
    t = g["acquired"].iloc[0]
    # rectangle autour des détections de la scène, avec une marge de 0,1°
    bbox = [g.lon.min() - 0.1, g.lat.min() - 0.1, g.lon.max() + 0.1, g.lat.max() + 0.1]
    print(f"{sid[:40]}…  {t:%Y-%m-%d %H:%M} UTC")
    a = ais_presence(bbox, t.normalize())
    a["scene_id"] = sid
    ais_frames.append(a)
    print(f"  {len(a)} lignes AIS reçues")

ais = pd.concat(ais_frames, ignore_index=True)
ais.to_csv(f"{OUT}/ais_raw.csv", index=False)
ais.head()

## 5. Garder, pour chaque navire AIS, sa position à l'heure du passage satellite

GFW donne la présence **heure par heure** (colonne `date`). Pour chaque navire, on garde l'heure la plus proche du passage. Un navire qui roule à 12 nœuds parcourt 22 km en une heure : on élargit donc le rayon de recherche selon sa vitesse AIS et l'écart de temps.

In [ ]:
import numpy as np

def haversine_km(lat1, lon1, lat2, lon2):
    # Distance sur la Terre entre deux points (km)
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

ais["t_hour"] = pd.to_datetime(ais["date"], utc=True, errors="coerce")   # début de l'heure AIS
acq = dets.groupby("scene_id")["acquired"].first()                       # heure de chaque passage
ais["acquired"] = ais["scene_id"].map(acq)
# écart (en heures) entre le milieu de l'heure AIS et le passage du satellite
ais["dt_h"] = ((ais.t_hour + pd.Timedelta(minutes=30)) - ais.acquired).abs().dt.total_seconds() / 3600

id_col = "vesselId" if "vesselId" in ais else "vessel_id"
ais_t = ais[ais.dt_h <= TIME_WINDOW_H + 0.5].copy()                       # heures voisines du passage
# une seule ligne par navire et par scène : l'heure la plus proche
ais_t = ais_t.sort_values("dt_h").drop_duplicates(["scene_id", id_col])

# Rayon de recherche : MATCH_KM + distance que le navire a pu parcourir pendant l'écart de temps
speed_kn = pd.to_numeric(ais_t.get("speed", 0), errors="coerce").fillna(0)
ais_t["radius_km"] = np.minimum(MATCH_KM + speed_kn * 1.852 * ais_t.dt_h, 15)

print(f"{len(ais_t)} navires AIS présents autour de l'heure des passages")
ais_t.groupby("scene_id").agg(navires_ais=(id_col, "nunique"), rayon_moyen_km=("radius_km", "mean")).round(2)

## 5 bis. Diagnostic : à quelle distance est l'AIS le plus proche ?

Avant de classer, on vérifie la méthode. Pour différentes fenêtres de temps et différents rayons, quelle part des échos radar a au moins un navire AIS à proximité ? Si la part grimpe beaucoup entre 2 et 10 km, c'est un décalage de position ; si elle grimpe avec la fenêtre de temps, c'est un décalage d'heure.

In [ ]:
rows = []
for win in (1, 3, 12):
    sub = ais[ais.dt_h <= win + 0.5]
    for R in (1, 2, 3, 5, 10):
        n_ok = 0
        for sid, g in dets.groupby("scene_id"):
            a = sub[sub.scene_id == sid]
            if a.empty:
                continue
            D = haversine_km(g.lat.values[:, None], g.lon.values[:, None], a.lat.values[None, :], a.lon.values[None, :])
            n_ok += int((D.min(1) <= R).sum())
        rows.append({"fenetre_h": win, "rayon_km": R, "part": round(n_ok / len(dets), 3)})
sens = pd.DataFrame(rows).pivot(index="rayon_km", columns="fenetre_h", values="part")
print("Part des échos radar ayant un navire AIS à moins de R km (colonnes : fenêtre ± heures)")
print(sens.to_string())

# Décalage systématique ? (médiane du vecteur écho -> AIS le plus proche, à moins de 5 km)
dets["nn_ais_km"] = np.nan
for sid, g in dets.groupby("scene_id"):
    a = ais_t[ais_t.scene_id == sid]
    if a.empty:
        continue
    D = haversine_km(g.lat.values[:, None], g.lon.values[:, None], a.lat.values[None, :], a.lon.values[None, :])
    j = D.argmin(1); nn = D.min(1)
    dets.loc[g.index, "nn_ais_km"] = nn
    near = nn < 5
    dn = (a.lat.values[j] - g.lat.values)[near] * 111.2
    de = (a.lon.values[j] - g.lon.values)[near] * 111.2 * np.cos(np.radians(6))
    print(f"{sid[17:32]}  échos {len(g):4d}  AIS {len(a):4d}  distance médiane {np.median(nn):6.2f} km  "
          f"décalage médian N {np.median(dn):+.2f} km, E {np.median(de):+.2f} km")

## 6. Le croisement

Chaque navire AIS ne peut « expliquer » qu'**une seule** détection radar : on associe d'abord les paires les plus proches. Sans ça, dans une rade chargée comme celle de Lagos, un seul navire AIS pourrait couvrir dix bateaux silencieux autour de lui.

Si GFW n'a encore **aucune** donnée AIS pour une scène (délai de publication), on ne lance pas d'alerte : la scène est marquée « AIS indisponible ».

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    # Distance sur la Terre entre deux points (km)
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

dets["ais_match"] = False
dets["ais_available"] = True
for c in ("ship_name", "flag", "ais_type", "mmsi"):
    dets[c] = None
dets["ais_dist_km"] = np.nan

for sid, g in dets.groupby("scene_id"):
    a = ais_t[ais_t.scene_id == sid]
    if a.empty:                                   # pas encore d'AIS publié pour ce passage
        dets.loc[g.index, "ais_available"] = False
        continue
    D = haversine_km(g.lat.values[:, None], g.lon.values[:, None],
                     a.lat.values[None, :], a.lon.values[None, :])     # matrice détections × AIS
    ok = D <= a.radius_km.values[None, :]                                # rayon propre à chaque navire
    pairs = sorted((D[i, j], i, j) for i, j in zip(*np.where(ok)))       # plus proches d'abord
    used_det, used_ais = set(), set()
    for dist, i, j in pairs:
        if i in used_det or j in used_ais:
            continue
        used_det.add(i); used_ais.add(j)
        idx = g.index[i]; row = a.iloc[j]
        dets.loc[idx, ["ais_match", "ais_dist_km"]] = [True, dist]
        dets.loc[idx, "ship_name"] = row.get("shipName")
        dets.loc[idx, "flag"] = row.get("flag")
        dets.loc[idx, "ais_type"] = row.get("vesselType")
        dets.loc[idx, "mmsi"] = row.get("mmsi")

# Objets fixes : un écho qui revient au même endroit (à FIXED_M près) sur au moins
# FIXED_MIN_PASSES passages différents est une plateforme pétrolière, une bouée, une épave
# ou un navire amarré en permanence, pas un navire « dark ».
P = dets[["lat", "lon"]].values
same_spot = haversine_km(P[:, 0][:, None], P[:, 1][:, None], P[:, 0][None, :], P[:, 1][None, :]) <= FIXED_M / 1000
codes = dets.scene_id.astype("category").cat.codes.values
dets["n_passes_same_spot"] = [len(set(codes[same_spot[i]])) for i in range(len(dets))]
dets["fixed"] = dets.n_passes_same_spot >= FIXED_MIN_PASSES

# Classement final
def category(r):
    if not r.ais_available:
        return "AIS indisponible"
    if r.ais_match:
        return "identifié (AIS)"
    if r.fixed:
        return "objet fixe"
    if r.vessel_length_m >= MIN_ALERT_LENGTH_M:
        return "ALERTE : dark ≥ 25 m"
    return "sans AIS, petit (< 25 m)"

dets["statut"] = dets.apply(category, axis=1)
summary = dets.groupby(["scene_id", "statut"]).size().unstack(fill_value=0)
dets.to_csv(f"{OUT}/vigiemer_detections_ais.csv", index=False)
summary

## 7. Les alertes

In [ ]:
alerts = dets[dets.statut.str.startswith("ALERTE")].sort_values("vessel_length_m", ascending=False)
alerts.to_csv(f"{OUT}/vigiemer_alertes.csv", index=False)
print(f"{len(alerts)} navires de 25 m ou plus sans AIS")
alerts[["acquired", "lat", "lon", "vessel_length_m", "vessel_speed_k",
        "is_fishing_vessel", "score"]].head(30).round(3)

## 8. Vignettes radar des alertes (contrôle visuel)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def crop_path(rel):
    # Le notebook 01 enregistre un chemin relatif « scène/crops/det_xxxx.png »
    if not isinstance(rel, str) or not rel:
        return None
    p = os.path.join(INPUT_ROOT, rel)
    return p if os.path.exists(p) else None

show = alerts.head(24)
if len(show):
    ncol = 8; nrow = -(-len(show) // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.2 * ncol, 2.5 * nrow), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), show.iterrows()):
        p = crop_path(r.crop)
        if p:
            ax.imshow(Image.open(p), cmap="gray")
        ax.set_title(f"{r.vessel_length_m:.0f} m", fontsize=10)
    plt.suptitle("Alertes : navires de 25 m ou plus sans AIS (vignettes ≈ 760 m de côté)")
    plt.tight_layout()
    plt.savefig(f"{OUT}/alertes_vignettes.png", dpi=110)
    plt.show()

## 9. Carte VigieMer

In [ ]:
import folium

COLORS = {"identifié (AIS)": "#30d158", "ALERTE : dark ≥ 25 m": "#ff3b30",
          "sans AIS, petit (< 25 m)": "#ff9f0a", "AIS indisponible": "#8e8e93", "objet fixe": "#5e5ce6"}
fmap = folium.Map(location=[dets.lat.mean(), dets.lon.mean()], zoom_start=8, tiles="OpenStreetMap")
for _, r in dets.iterrows():
    if r.ais_match:
        title = f"<b>{r.ship_name or 'Navire AIS'}</b><br>Pavillon : {r.flag}<br>Type : {r.ais_type}"
    else:
        title = f"<b>{r.statut}</b>"
    popup = (f"{title}<br>{r.acquired:%Y-%m-%d %H:%M} UTC<br>"
             f"Longueur estimée ≈ {r.vessel_length_m:.0f} m<br>"
             f"{r.lat:.4f}, {r.lon:.4f}")
    folium.CircleMarker([r.lat, r.lon], radius=6 if r.statut.startswith("ALERTE") else 4,
                        color=COLORS[r.statut], fill=True, fill_opacity=0.9,
                        popup=folium.Popup(popup, max_width=260)).add_to(fmap)
legend = "".join(f"<div><span style='color:{c}'>●</span> {k}</div>" for k, c in COLORS.items())
fmap.get_root().html.add_child(folium.Element(
    f"<div style='position:fixed;bottom:24px;left:24px;z-index:9999;background:#1c1c1e;"
    f"color:#fff;padding:10px 14px;border-radius:8px;font:13px sans-serif'>"
    f"<b>VigieMer</b>{legend}</div>"))
fmap.save(f"{OUT}/carte_vigiemer.html")
fmap

## 10. Contrôle qualité avec les détections radar de GFW

Global Fishing Watch publie aussi ses propres détections radar (sur les mêmes images Sentinel-1), environ 5 jours après le passage. Comparer les comptes est un test indépendant de notre détecteur.

In [ ]:
def gfw_sar(bbox, day):
    lon0, lat0, lon1, lat1 = bbox
    polygon = {"type": "Polygon",
               "coordinates": [[[lon0, lat0], [lon1, lat0], [lon1, lat1], [lon0, lat1], [lon0, lat0]]]}
    params = {"datasets[0]": "public-global-sar-presence:latest",
              "date-range": f"{day:%Y-%m-%d},{day + pd.Timedelta(days=1):%Y-%m-%d}",
              "temporal-resolution": "ENTIRE", "spatial-resolution": "HIGH",
              "group-by": "VESSEL_ID", "format": "JSON"}
    for attempt in range(4):
        r = requests.post(REPORT_URL, params=params, json={"geojson": polygon}, headers=HEADERS, timeout=180)
        if r.status_code != 429:
            break
        time.sleep(20)
    if not r.ok:
        return None
    return pd.DataFrame(flatten_entries(read_report(r)))

sar_stats = {}
for sid, g in dets.groupby("scene_id"):
    bbox = [g.lon.min() - 0.1, g.lat.min() - 0.1, g.lon.max() + 0.1, g.lat.max() + 0.1]
    sar = gfw_sar(bbox, g.acquired.iloc[0].normalize())
    n_gfw = None
    if sar is not None and len(sar):
        n_gfw = int(pd.to_numeric(sar["detections"], errors="coerce").sum()) if "detections" in sar else len(sar)
    sar_stats[sid] = {"vigiemer": int(len(g)), "gfw_sar": n_gfw}
    print(f"{sid[:45]}  VigieMer : {len(g):4d}   GFW SAR : {n_gfw if n_gfw is not None else 'pas encore publié'}")

## 11. Export pour l'application VigieMer

Un seul fichier `vigiemer_app_data.json` : les scènes (heure, emprise, aperçu radar), toutes les détections avec leur statut, et les vignettes radar (JPEG 64 px) des alertes et d'un échantillon de navires identifiés.

In [ ]:
import base64, io, glob

def crop_b64(rel):
    p = crop_path(rel)
    if not p:
        return None
    im = Image.open(p).convert("L").resize((64, 64), Image.BILINEAR)
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=72)
    return base64.b64encode(buf.getvalue()).decode()

# Vignettes : toutes les alertes, puis les 40 navires identifiés les plus sûrs, puis 20 petits sans AIS
want = set(alerts.head(150).index)
want |= set(dets[dets.statut == "identifié (AIS)"].sort_values("score", ascending=False).head(40).index)
want |= set(dets[dets.statut.str.startswith("sans AIS")].sort_values("score", ascending=False).head(20).index)
want |= set(dets[dets.statut == "objet fixe"].sort_values("vessel_length_m", ascending=False).head(20).index)

def clean(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    return v

records = []
for i, r in dets.iterrows():
    records.append({
        "id": int(i), "scene": r.scene_id, "t": r.acquired.strftime("%Y-%m-%dT%H:%MZ"),
        "lat": round(float(r.lat), 5), "lon": round(float(r.lon), 5),
        "len": round(float(r.vessel_length_m), 1), "wid": round(float(r.vessel_width_m), 1),
        "spd": round(float(r.vessel_speed_k), 1), "hdg": float(r.heading_deg),
        "fish": round(float(r.is_fishing_vessel), 3), "score": round(float(r.score), 3),
        "status": r.statut, "name": clean(r.ship_name), "flag": clean(r.flag),
        "type": clean(r.ais_type), "dist": clean(round(float(r.ais_dist_km), 2) if r.ais_match else None),
        "nn": clean(round(float(r.nn_ais_km), 2) if pd.notna(r.nn_ais_km) else None),
        "npass": int(r.n_passes_same_spot),
        "img": crop_b64(r.crop) if i in want else None,
    })

# Aperçu radar de chaque scène (bande VV réduite) pour l'afficher sur la carte de l'application.
# L'image est en Web Mercator, alignée nord-sud : il suffit de connaître ses 4 coins.
def quicklook_b64(sid, width=400):
    hits = glob.glob(f"/kaggle/input/**/{sid}/quicklook_vv.npy", recursive=True)
    if not hits:
        return None
    q = np.load(hits[0]).astype(np.float32)
    valid = q > 0                                    # 0 = hors de la fauchée radar
    if valid.sum() < 100:
        return None
    lo, hi = np.percentile(q[valid], [2, 99.5])      # étirement du contraste
    x = np.clip((q - lo) / max(hi - lo, 1), 0, 1) ** 0.7 * 255
    x[~valid] = 0
    im = Image.fromarray(x.astype(np.uint8))
    im = im.resize((width, max(1, round(im.height * width / im.width))), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=45, optimize=True)
    return base64.b64encode(buf.getvalue()).decode()

def ql_bounds(corners):
    # coins [lon, lat] -> [[sud, ouest], [nord, est]] (format Leaflet)
    if not corners:
        return None
    lons = [c[0] for c in corners]; lats = [c[1] for c in corners]
    return [[round(min(lats), 5), round(min(lons), 5)], [round(max(lats), 5), round(max(lons), 5)]]

scenes_out = []
for sid, g in dets.groupby("scene_id"):
    m = scene_meta.get(sid, {})
    scenes_out.append({"id": sid, "t": g.acquired.iloc[0].strftime("%Y-%m-%dT%H:%MZ"),
                       "footprint": m.get("footprint"), "bounds": m.get("bounds"),
                       "ql": quicklook_b64(sid), "ql_bounds": ql_bounds(m.get("bounds")),
                       "counts": g.statut.value_counts().to_dict(),
                       "gfw_sar": sar_stats.get(sid, {}).get("gfw_sar"),
                       "ais_vessels": int(ais_t[ais_t.scene_id == sid][id_col].nunique())})

app = {"generated": pd.Timestamp.now("UTC").strftime("%Y-%m-%dT%H:%MZ"),
       "params": {"match_km": MATCH_KM, "window_h": TIME_WINDOW_H,
                  "min_alert_len": MIN_ALERT_LENGTH_M, "min_score": MIN_SCORE,
                  "fixed_m": FIXED_M, "fixed_min_passes": FIXED_MIN_PASSES},
       "sensitivity": sens.reset_index().to_dict(orient="list"),
       "scenes": scenes_out, "detections": records}
with open(f"{OUT}/vigiemer_app_data.json", "w") as f:
    json.dump(app, f, separators=(",", ":"))
size_kb = os.path.getsize(f"{OUT}/vigiemer_app_data.json") / 1024
print(f"vigiemer_app_data.json : {len(records)} détections, {len(scenes_out)} scènes, "
      f"{sum(1 for x in records if x['img'])} vignettes, {size_kb:.0f} Ko")
print(dets.statut.value_counts().to_string())

## Résultat

- `vigiemer_detections_ais.csv` : toutes les détections, avec statut, nom et pavillon quand l'AIS est trouvé
- `vigiemer_alertes.csv` et `alertes_vignettes.png` : les navires « dark » de 25 m ou plus
- `carte_vigiemer.html` : la carte
- `vigiemer_app_data.json` : les données de l'application VigieMer

**Limites honnêtes** (à dire dans le dossier) : une détection sans AIS n'est pas une preuve d'activité illégale. L'AIS peut être brouillé ou mal capté, le navire peut être un bateau militaire ou de service, et le radar peut confondre un navire avec une bouée ou une petite plateforme. VigieMer **priorise** les navires à contrôler, il ne juge pas.